In [1]:
%load_ext autoreload
%autoreload 2

%env CUPY_ACCELERATORS=cub
%env TENSORLY_BACKEND=numpy

env: CUPY_ACCELERATORS=cub
env: TENSORLY_BACKEND=numpy


In [2]:
from moabb.paradigms import MotorImagery
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

datasets = [
    AlexMI(),
    #BNCI2014_001(),
    #PhysionetMI(),
    #Schirrmeister2017(),
    #Weibo2014(),
    #Zhou2016()
]

n_classes=3
sfreq=250

In [3]:
import copy
from sklearn.base import clone
import dask
import os
import tensorly as tl
from classification_mi import stf_transform
from sklearn.preprocessing import FunctionTransformer

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

def eval_moabb_within_session(dataset, subject, pipe):
    subj_dataset = copy.deepcopy(dataset)

    events = list(subj_dataset.event_id.keys())[:n_classes]
    paradigm = MotorImagery(events=events, n_classes=n_classes, resample=sfreq)

    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=False,
        random_state=42,
        n_jobs=5,
        suffix=f'bttda_dask_dataset-{dataset.code}_subject-{subject}_pipe-{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process(
        {pipe:clone(pipelines[pipe])},
        postprocess_pipeline= FunctionTransformer(stf_transform)
    )





In [4]:
from classification_mi import get_pipelines_mi
pipelines = get_pipelines_mi()
pipelines


{'HODA': Pipeline(steps=[('zscore1', ZScore()),
                 ('bttda',
                  BTTDACV(clf=Pipeline(steps=[('functiontransformer',
                                               FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x151bb8729800>)),
                                              ('pca', PCA(whiten=True)),
                                              ('selectfcutoff', SelectFCutoff()),
                                              ('lineardiscriminantanalysis',
                                               LinearDiscriminantAnalysis(shrinkage='auto',
                                                                          solver='lsqr'))]),
                          cv=StratifiedKFold(n_splits...
                                       'tol': 0.0001, 'verbose': False},
                          max_n_blocks=1,
                          thetas=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8,
                                  0.9, 1.0])),
              

In [5]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd
from hpc import create_cluster, create_client, TIMEOUT

#import dask.config
#
#dask.config.set({
#    "distributed.scheduler.worker-saturation": 0.25
#})

import warnings
#warnings.filterwarnings("ignore", category=RuntimeWarning)
#warnings.simplefilter("error", category=UserWarning)
import pdb
%pdb on

with create_cluster(cluster='wice_sapphirerapids') as cluster, create_client(cluster) as client:
    results = []
    for dataset in datasets:
        print(f'Benchmarking on dataset {dataset.code}...')
        job_args = []
        for subject in dataset.subject_list:
            for pipe in pipelines.keys():
                job_args.append((dataset, subject,pipe))    
        with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
            n_jobs = len(dataset.subject_list)*len(pipelines)
            results += Parallel(n_jobs=n_jobs, verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

Automatic pdb calling has been turned ON
Benchmarking on dataset AlexandreMotorImagery...


[Parallel(n_jobs=24)]: Using backend DaskDistributedBackend with 90 concurrent workers.
[Parallel(n_jobs=24)]: Done  20 out of  24 | elapsed: 13.2min remaining:  2.6min
[Parallel(n_jobs=24)]: Done  24 out of  24 | elapsed: 13.8min finished


In [8]:
results.to_csv('results/moabb_mi_new.csv')
results

,score,time,samples,subject,session,channels,n_sessions,dataset,pipeline
0,0.350000,85.385887,60.0,1,0,16,1,AlexandreMotorImagery,HODA
1,0.416667,122.247917,60.0,1,0,16,1,AlexandreMotorImagery,PARAFACDA
2,0.433333,150.179779,60.0,1,0,16,1,AlexandreMotorImagery,BTTDA
3,0.466667,65.811691,60.0,2,0,16,1,AlexandreMotorImagery,HODA
4,0.450000,106.367889,60.0,2,0,16,1,AlexandreMotorImagery,PARAFACDA
5,0.533333,158.969818,60.0,2,0,16,1,AlexandreMotorImagery,BTTDA
6,0.366667,55.798233,60.0,3,0,16,1,AlexandreMotorImagery,HODA
7,0.616667,99.426003,60.0,3,0,16,1,AlexandreMotorImagery,PARAFACDA
8,0.516667,149.284821,60.0,3,0,16,1,AlexandreMotorImagery,BTTDA
9,0.650000,80.726448,60.0,4,0,16,1,AlexandreMotorImagery,HODA


In [9]:
results = pd.read_csv('results/moabb_mi_new.csv')

In [10]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate(['mean', 'std'])

mean       std
dataset               pipeline                     
AlexandreMotorImagery BTTDA      0.518750  0.130456
                      HODA       0.493750  0.147179
                      PARAFACDA  0.516667  0.109472

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

In [ ]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    fig.add_shape(
        type="line",
        x0=0, y0=0.0, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig